# 04 · Lapse prediction, 12-month value and owner segments
Lapse = no accessory, referral or care-hub activity in months 7-12. Features are a month-6 snapshot (no leakage).

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from neatnest import retention
cu = pd.read_csv('../data/customers.csv')
print(cu.shape, 'lapse rate', round(cu.lapsed_m7_12.mean(), 4))

(8000, 18) lapse rate 0.3114


In [2]:
lm = retention.lapse_models(cu)
lm['metrics'].round(3)

,auc,accuracy,precision,recall,f1,cv_auc
Logistic (balanced),0.782,0.719,0.538,0.697,0.607,0.793
Random Forest,0.779,0.722,0.541,0.695,0.609,0.785


In [3]:
lm['importance'].head(10).round(4)

support_tickets_m1_6        0.0397
nps_m3                      0.0307
whatsapp_open_rate          0.0195
product_issue_reported      0.0164
accessory_spend_m1_6        0.0057
delivery_delay_days         0.0050
referrals_m1_6              0.0032
accessory_orders_m1_6       0.0023
care_hub_logins_m1_6        0.0013
discount_pct_first_order    0.0012
dtype: float64

In [4]:
lm['lift'].round(3)

,customers,lapsers,rate,cum_capture
decile,,,,
1,160,137,0.856,0.275
2,160,79,0.494,0.434
3,160,69,0.431,0.572
4,160,61,0.381,0.695
5,160,39,0.244,0.773
6,160,31,0.194,0.835
7,160,27,0.169,0.890
8,160,29,0.181,0.948
9,160,20,0.125,0.988


In [5]:
vm = retention.value_models(cu)
print('max VIF', round(vm['vif_max'], 2))
vm['metrics'].round(3)

max VIF 2.05


,r2,rmse,mae,n_zero_coef
OLS,0.928,256.553,202.610,0.0
Ridge,0.928,256.556,202.623,0.0
Lasso,0.928,256.531,202.827,5.0


In [6]:
sg = retention.segments(cu)
print({k: round(v, 3) for k, v in sg['silhouette'].items()})
sg['profile']

{2: 0.218, 3: 0.208, 4: 0.161, 5: 0.174, 6: 0.185, 7: 0.186}


,customers,lapse_rate,value_12m,care_hub_logins_m1_6,whatsapp_open_rate,accessory_spend_m1_6,referrals_m1_6,support_tickets_m1_6,nps_m3,share
Quiet One-timers,2617,0.411,2059.498,1.442,29.522,122.117,0.228,0.413,5.879,0.327
Steady Replenishers,2929,0.203,2230.900,2.319,50.902,151.912,0.249,0.413,8.377,0.366
Service-scarred,1085,0.647,2109.982,2.065,42.326,187.272,0.218,2.633,6.076,0.136
Nest Advocates,1369,0.087,3620.430,4.757,62.318,538.839,1.137,0.629,8.561,0.171
